# 21. pgvector HNSW — 빌드 · ef_search · m

11번에서 numpy 로 만든 것을 pgvector 로 한다. 파라미터 이름이 그대로다.

| 시점 | 파라미터 | 기본값 | 뜻 |
|---|---|---|---|
| 빌드 | `m` | 16 | 층마다 이웃 수 (0층은 2m) |
| 빌드 | `ef_construction` | 64 | 빌드 때 이웃 후보를 찾는 폭 |
| 검색 | `hnsw.ef_search` | 40 | 검색 때 들고 다니는 후보 수 |

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결 · 남은 인덱스 정리 |
| 3 | 빌드 |
| 4 | 연산자와 인덱스의 짝 |
| 5 | ef_search — recall 과 지연시간 |
| 6 | 함정 — ef_search 보다 많이 달라면 |
| 7 | 빌드 파라미터 m |
| 8 | 실시간 추가 |

**Run → Run All Cells** 로 한 번에 실행하거나, 셀을 하나씩 **Shift + Enter**

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"

### 셀 2 · 연결 · 남은 인덱스 정리

In [ ]:
# 셀 2 · 연결 · 남은 인덱스 정리
from labutil import *

conn = connect(DB_URL)
os.makedirs("results", exist_ok=True)
for ds in ["clu", "iso"]:                     # 다른 노트북이 남긴 벡터 인덱스를 지운다 (planner 가 그걸 쓰지 않게)
    for kind in ["hnsw", "ivf", "hnsw_m"]:
        conn.execute(f"DROP INDEX IF EXISTS vec_{ds}_{kind}")

## 1. 빌드

### 셀 3 · 빌드

In [ ]:
# 셀 3 · 빌드
build = {}
for ds in ["clu", "iso"]:
    with timer() as t:
        conn.execute(f"CREATE INDEX vec_{ds}_hnsw ON vec_{ds} USING hnsw (emb vector_l2_ops) "
                     "WITH (m = 16, ef_construction = 64)")
    build[ds] = {"빌드(s)": t["s"], "인덱스 MB": rel_size(conn, f"vec_{ds}_hnsw"), "테이블 MB": rel_size(conn, f"vec_{ds}")}
build = pd.DataFrame(build).T

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
build["빌드(s)"].plot.barh(ax=axes[0], color=["#FF7043", "#3D484A"]); axes[0].set_title("빌드 시간 (s) — 10만 × 128차원, 병렬")
build[["테이블 MB", "인덱스 MB"]].plot.barh(ax=axes[1], color=["#9aa5a8", "#009CA6"]); axes[1].set_title("크기 (MB)")
plt.tight_layout(); plt.show()
build.round(1)

**관찰:** HNSW 인덱스는 **벡터를 통째로 복사해 들고 있고** 그 위에 이웃 목록까지 얹는다 → 테이블보다 크다.
슬라이드 "HNSW 는 그래프 + 벡터가 전부 RAM 상주(메모리 큼)".
무구조형이 빌드가 더 오래 걸린다 — 이웃 후보를 찾는 탐색 자체가 어렵기 때문.

## 2. 인덱스를 타는가 — 거리 연산자와 인덱스는 짝이 맞아야 한다

### 셀 4 · 연산자와 인덱스의 짝

In [ ]:
# 셀 4 · 연산자와 인덱스의 짝
qv = q(conn, "SELECT emb FROM vec_query WHERE ds='clu' AND qid=0")["emb"][0]
for op, name in [("<->", "L2 (인덱스와 같은 척도)"), ("<=>", "코사인 (다른 척도)")]:
    print(f"── ORDER BY emb {op} q  : {name}")
    print(explain(conn, f"SELECT id FROM vec_clu ORDER BY emb {op} %s LIMIT 10", (qv,)), "\n")

**관찰:** `vector_l2_ops` 로 만든 인덱스는 `<->` 에만 쓰인다. `<=>`(코사인)로 물으면 Seq Scan 으로 돌아간다.
HNSW 알고리즘은 어떤 척도에도 쓸 수 있지만, **만들어진 인덱스는 빌드할 때 고른 척도 하나에 묶인다**.

## 3. ef_search — recall 과 지연시간

### 셀 5 · ef_search — recall 과 지연시간

In [ ]:
# 셀 5 · ef_search — recall 과 지연시간
conn.execute("SET enable_indexscan = off")
seq = {ds: pg_bench(conn, f"vec_{ds}", ds, n=50) for ds in ["clu", "iso"]}   # 전수 비교 기준선
conn.execute("RESET enable_indexscan")

rows = []
for ds in ["clu", "iso"]:
    for ef in [10, 20, 40, 80, 160, 320]:
        conn.execute(f"SET hnsw.ef_search = {ef}")
        rows.append({"ds": ds, "ef_search": ef, **pg_bench(conn, f"vec_{ds}", ds)})
conn.execute("RESET hnsw.ef_search")
curve = pd.DataFrame(rows)
curve.to_csv("results/hnsw_curve.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), layout="constrained")
for ds, color, name in [("clu", "#FF7043", "군집형"), ("iso", "#3D484A", "무구조형")]:
    g = curve[curve.ds == ds]
    axes[0].plot(g["ms"], g["recall"], "-o", color=color, label=name)
    for _, r in g.iterrows():
        axes[0].annotate(int(r.ef_search), (r.ms, r.recall), textcoords="offset points", xytext=(4, -10), fontsize=8)
    axes[0].axvline(seq[ds]["ms"], color=color, ls=":", label=f"{name} 전수 비교 {seq[ds]['ms']:.0f}ms")
    axes[1].plot(g["ef_search"], g["recall"], "-o", color=color, label=name)
axes[0].set_xscale("log"); axes[0].set_xlabel("질의 지연시간 (ms, 중앙값)   ※ 점 옆 숫자 = ef_search")
axes[0].set_ylabel("recall@10"); axes[0].legend(fontsize=8); axes[0].set_title("속도 ↔ 정확도")
axes[1].set_xscale("log", base=2); axes[1].set_xlabel("hnsw.ef_search"); axes[1].set_ylabel("recall@10")
axes[1].axvline(40, color="gray", ls="--"); axes[1].text(40, 0.5, " 기본값 40", color="gray")
axes[1].legend(); axes[1].set_title("ef_search ↑ → recall ↑")
plt.show()
curve.pivot(index="ef_search", columns="ds", values=["recall", "ms"]).round(3)

**관찰:**
- 군집형은 기본값(40)에서 recall 약 0.9, 무구조형은 약 0.4 — 같은 인덱스 설정이라도 데이터에 따라 크게 다르다
- 전수 비교(점선)보다 10배 이상 빠르다. ef_search 를 올리면 recall 은 오르고 지연시간도 는다
- 1ms 아래의 지연시간에는 네트워크 왕복·잡음이 섞인다. 경향만 본다
- 실제 임베딩은 군집형에 가깝다. 그래도 **내 데이터에서 recall 을 재 보고** ef_search 를 정해야 한다

## 4. 함정: ef_search 보다 많이 달라고 하면?

### 셀 6 · 함정 — ef_search 보다 많이 달라면

In [ ]:
# 셀 6 · 함정 — ef_search 보다 많이 달라면
rows = []
for ef in [10, 40, 100, 200]:
    conn.execute(f"SET hnsw.ef_search = {ef}")
    n = len(q(conn, "SELECT id FROM vec_clu ORDER BY emb <-> %s LIMIT 100", (qv,)))
    rows.append({"ef_search": ef, "LIMIT 100 에 돌아온 행": n})
conn.execute("RESET hnsw.ef_search")
t = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(t["ef_search"].astype(str), t["LIMIT 100 에 돌아온 행"], color="#009CA6")
ax.axhline(100, color="#FF7043", ls="--", label="요청한 LIMIT 100")
ax.set_xlabel("hnsw.ef_search"); ax.set_ylabel("받은 행 수"); ax.legend(); ax.set_title("HNSW 는 최대 ef_search 개만 돌려준다")
plt.show()

**관찰:** `LIMIT 100` 인데 `ef_search=40` 이면 **40행만** 온다. 오류도 경고도 없다.
후보를 넉넉히 뽑아 재정렬(리랭킹)하려면 ef_search 를 LIMIT 이상으로 올려야 한다.
(pgvector 0.8 의 `hnsw.iterative_scan` 을 켜면 더 찾아온다 → 4-1 실습6)

## 5. 빌드 파라미터 m — 크기 · 빌드 시간 · recall

### 셀 7 · 빌드 파라미터 m

In [ ]:
# 셀 7 · 빌드 파라미터 m
rows = []
for m in [4, 8, 16, 32]:
    conn.execute("DROP INDEX IF EXISTS vec_clu_hnsw_m")
    with timer() as t:
        conn.execute(f"CREATE INDEX vec_clu_hnsw_m ON vec_clu USING hnsw (emb vector_l2_ops) WITH (m = {m}, ef_construction = 64)")
    conn.execute("DROP INDEX vec_clu_hnsw")          # 비교할 인덱스만 남긴다
    r = pg_bench(conn, "vec_clu", "clu")             # ef_search 기본값 40
    rows.append({"m": m, "빌드(s)": t["s"], "인덱스 MB": rel_size(conn, "vec_clu_hnsw_m"), **r})
    conn.execute("CREATE INDEX vec_clu_hnsw ON vec_clu USING hnsw (emb vector_l2_ops)")   # 원래 인덱스 복구
conn.execute("DROP INDEX IF EXISTS vec_clu_hnsw_m")
mres = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, col, color in zip(axes, ["인덱스 MB", "빌드(s)", "recall", "ms"], ["#009CA6", "#3D484A", "#FF7043", "#9aa5a8"]):
    ax.bar(mres["m"].astype(str), mres[col], color=color)
    for i, v in enumerate(mres[col]):
        ax.text(i, v, f"{v:.3g}", ha="center", va="bottom", fontsize=9)
    ax.set_xlabel("m"); ax.set_title(col if col != "ms" else "지연시간 ms")
axes[2].set_ylim(0, 1.05)
plt.suptitle("m ↑ → 이웃 목록이 커져 인덱스·빌드 ↑, recall ↑ (ef_search=40)", y=1.03); plt.tight_layout(); plt.show()
mres.round(3)

**관찰:** 인덱스 크기는 m 에 따라 커지지만 **벡터 복사본이 대부분**이라 두 배까지는 안 커진다.
m 이 너무 작으면(4) 그래프가 약해 recall 이 떨어진다 — 10번의 "섬" 현상과 같다.

## 6. 실시간 추가 — HNSW 는 INSERT 를 바로 반영한다

### 셀 8 · 실시간 추가

In [ ]:
# 셀 8 · 실시간 추가
rng = np.random.default_rng(7)
Xc = q(conn, "SELECT emb FROM vec_clu ORDER BY id LIMIT 500")["emb"]
new = np.stack(Xc.to_numpy()) + rng.normal(0, 0.3, (500, 128)).astype(np.float32)
with timer() as t, conn.cursor() as cur:
    for i, v in enumerate(new):
        cur.execute("INSERT INTO vec_clu (id, cluster, emb) VALUES (%s, -1, %s)", (1_000_000 + i, v))
print(f"500건을 한 건씩 INSERT (인덱스 갱신 포함): {t['s']:.2f}s → 건당 {1000 * t['s'] / 500:.2f}ms")
hit = q(conn, "SELECT id FROM vec_clu ORDER BY emb <-> %s LIMIT 1", (new[0],))["id"][0]
print("방금 넣은 벡터로 검색하면 1위:", hit)
conn.execute("DELETE FROM vec_clu WHERE id >= 1000000")
_ = conn.execute("VACUUM vec_clu")  # 삭제된 노드를 그래프에서 정리 (HNSW 삭제는 표시 후 VACUUM 이 수선)

**관찰:** 새 벡터는 들어가는 즉시 검색된다 (슬라이드 "HNSW 는 추가가 쉬움, 실시간 색인 적합").
삭제는 표시만 해 두고 `VACUUM` 때 이웃을 다시 이어 준다 — 대량 삭제가 잦으면 재빌드가 낫다.